# Tiny GPT Reader on Kaggle: the pointing (span) Reader

Fine-tunes the pretrained Reader to point at answers, on Kaggle's free GPU.

**Before running:**
1. **Settings** (right panel): **Accelerator: GPU T4 x2** (or P100), **Internet: on**. Both need a phone-verified Kaggle account.
2. **Add Input > Datasets > Your Datasets**: add the dataset with `ckpt.pt` and `tokenizer.json` (from `MyDrive/tiny-gpt-reader/for-kaggle` on Google Drive).

Run the cells one by one to try it. For the full run, use **Save Version > Save & Run All**: it keeps running with the browser closed, and the results appear in the notebook's **Output** tab (`span.zip`).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Find the pretrained Reader

In [ ]:
import glob, os
found = glob.glob("/kaggle/input/**/ckpt.pt", recursive=True)
assert found, "No ckpt.pt under /kaggle/input: add your dataset (Add Input > Datasets)"
INIT = os.path.dirname(found[0])
assert os.path.exists(f"{INIT}/tokenizer.json"), "tokenizer.json must sit next to ckpt.pt"
print("pretrained Reader:", INIT)

## Get the code

In [ ]:
os.chdir("/kaggle/working")
if os.path.exists("tiny-gpt"):
    !cd tiny-gpt && git pull --ff-only
else:
    !git clone --depth 1 https://github.com/Nenethsu/tiny-gpt.git
os.chdir("/kaggle/working/tiny-gpt")
!git log --oneline -1
!python bpe.py test

## Smoke test on the real model (about 3 minutes)
40 steps on 2,000 questions: checks everything runs on this machine before the long run.

In [ ]:
!python finetune_span.py --preset smoke --init "$INIT" --out /kaggle/working/span-smoke

## Fine-tune the pointing Reader (about 25-40 minutes)

In [ ]:
!python finetune_span.py --init "$INIT" --out /kaggle/working/span

## Agreement tables

In [ ]:
!python evaluate.py --qa /kaggle/working/span --from-predictions

## Pack the results
`span.zip` (checkpoint, tokenizer, logs, exam answers) appears in the Output tab after a Save & Run All.

In [ ]:
!rm -rf /kaggle/working/span-smoke
!cd /kaggle/working && zip -qr span.zip span && ls -la span.zip